# 필요한 라이브러리

In [10]:
import pandas as pd
import urllib.request
from bs4 import BeautifulSoup
import requests
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import undetected_chromedriver
import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
import numpy as np
import warnings
import os
import xgboost
from xgboost import XGBRegressor, XGBClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, f1_score, accuracy_score, recall_score, precision_score
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, accuracy_score
from sklearn.model_selection import GridSearchCV

# 팀별 url 주소 & 팀별 선수 명단

In [2]:
# matchlist url
Team_S12_http_dict = {
    
    'T1':'https://gol.gg/teams/team-matchlist/1438/split-ALL/tournament-ALL/',
    'Gen.G':'https://gol.gg/teams/team-matchlist/1439/split-ALL/tournament-ALL/',
    'DWG_KIA':'https://gol.gg/teams/team-matchlist/1412/split-ALL/tournament-ALL/',
    'DRX':'https://gol.gg/teams/team-matchlist/1413/split-ALL/tournament-ALL/',
    'KT':'https://gol.gg/teams/team-matchlist/1417/split-ALL/tournament-ALL/',
    'Liiv_SB':'https://gol.gg/teams/team-matchlist/1418/split-ALL/tournament-ALL/',
    'NS':'https://gol.gg/teams/team-matchlist/1419/split-ALL/tournament-ALL/',
    'HLE':'https://gol.gg/teams/team-matchlist/1416/split-ALL/tournament-ALL/',
    'KDF':'https://gol.gg/teams/team-matchlist/1411/split-ALL/tournament-ALL/',
    'BRO':'https://gol.gg/teams/team-matchlist/1414/split-ALL/tournament-ALL/'
    
}

In [ ]:
# 팀 구성원 리스트
Team_Player_dict = {
    
    "T1": ["Zeus", "Oner", "Faker", "Gumayusi", "Keria"],
    "Gen.G": ["Doran", "Peanut", "Chovy", "Ruler", "Lehends"],
    "DWG_KIA": ["Nuguri", "Canyon", "ShowMaker", "deokdam", "Kellin","Burdol"],
    "DRX": ["kingen", "Pyosik", "Zeka", "Deft", "BeryL","Juhan"],
    "KT": ["Rascal", "Cuzz", "Vicla", "Aria", "Aiming", "Life"],
    "Liiv_SB": ["Dove", "Croco", "Clozer", "Prince", "Kael", "Envyy", "Ice"],
    "NS": ["Canna", "Dread", "Bdd", "Ghost", "Effort", "Peter", "SnowFlower"],
    "HLE": ["DuDu", "OnFleek", "Karis", "Hans SamD", "Vsta", "Cheoni"],
    "KDF": ["Kiin", "Ellim", "FATE", "Teddy", "Hoit", "Moham"],
    "BRO": ["Morgan", "UmTi", "Lava", "Hena", "Delight", "Sw0rd"]
    
}

In [ ]:
# 팀 구성원 별 데이터프레임 담을 딕셔너리
Team_dict = {
    
    'T1':{},
    'Gen.G':{},
    'DWG_KIA':{},
    'DRX':{},
    'KT':{},
    'Liiv_SB':{},
    'NS':{},
    'HLE':{},
    'KDF':{},
    'BRO':{}
    
}

In [ ]:
# 모든 팀 리스트
Team_list = ['T1','Gen.G','DWG_KIA','DRX','KT','Liiv_SB','NS','HLE','KDF','BRO']

In [ ]:
# 각 팀의 result 이름을 담은 딕셔너리
# 표기가 다름

Team_result_dict={'T1':'T1',
                  'Gen.G':'GEN',
                  'DWG_KIA':'DK',
                  'DRX':'DRX',
                  'KT':'KT',
                  'Liiv_SB':'LSB',
                  'NS':'NS',
                  'HLE':'HLE',
                  'KDF':'KDF',
                  'BRO':'BRO'}

## 팀별 각각의 데이터를 csv로 저장하는 코드

In [ ]:
## 함수 제작
def DataCollector(TeamName):
    
    driver = undetected_chromedriver.Chrome()
    # T1의 모든 시즌 데이터 있는 웹사이트
    url = Team_S12_http_dict[TeamName]

    driver.get(url)

    time.sleep(1)

    # 버튼 클릭
    # button = driver.find_element(By.ID, 'export-btn')
    button = driver.find_element(By.CLASS_NAME, 'btn.btn-gol')
    button.click()

    time.sleep(0.5)

    # 클립보드 복사
    copied_df = pd.read_clipboard(sep='\t')

    # 데이터 정제
    copied_df = copied_df.reset_index()
    copied_df = copied_df.rename(columns = pd.Series(copied_df.iloc[0,:]))[1:]

    driver.close()
    
    TeamName_ = Team_result_dict[TeamName]



    # 데이터 정제 및 변수로 인덱싱
    result_gameclass_df = copied_df.iloc[:,[0,-1]]

    # 새로운 칼럼 생성(복사)
    result_gameclass_df['Result Score'] = result_gameclass_df[f'{TeamName_} Result']
    result_gameclass_df['Tournament Score'] = result_gameclass_df['Tournament']

    # 승패를 숫자로 변경
    for i in range(result_gameclass_df.__len__()):
        result_gameclass_df['Result Score'].iloc[i] = np.where(result_gameclass_df[f'{TeamName_} Result'].iloc[i] == 'LOSS', 0, 1)

    # 경기종류에 따라 일정 숫자로 바꿈
    for i in range(result_gameclass_df.__len__()):
        result_gameclass_df['Tournament Score'].iloc[i] = np.where(result_gameclass_df['Tournament'].iloc[i][:3] == 'LCK', 1,
                                                                   np.where(result_gameclass_df['Tournament'].iloc[i][:3] =='MSI', 2, 2.5))

    # P 값 (승패[M] * 경기중요도[I] * Elo[E])
    result_gameclass_df['P value'] = result_gameclass_df['Result Score'] * result_gameclass_df['Tournament Score']*(2000-1623)/100


    ## 선수별 데이터 담기


    # url = Team_S12_http_dict[TeamName]

    headers = {'Accept-Language': 'ko_KR,en;q=0.8',
               'User-Agent': ('Mozilla/5.0 (Linux; Android 6.0; Nexus 5 Build/MRA58N) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/78.0.3904.70 Mobile Safari/537.36')}

    ## request 모듈

    html = requests.get(url, headers = headers)
    soup = BeautifulSoup(html.text)  # html.text 로 넣어야됨


    # td 먼저 분리
    td_tags = soup.find_all('td', class_='text-center')


    # a 태그의 href 길이를 보고 분리
    # links에 각 링크들을 저장
    links = []
    for td in td_tags:
        a_tag = td.find('a', href=True)
        if a_tag:
            if len(a_tag['href']) == 30:
                links.append(a_tag['href'])



    # undetected_chromedriver로 크롬 열기            
    driver = undetected_chromedriver.Chrome()

    # 데이터 담을 그릇
    Team_Player_list = Team_Player_dict[TeamName]
    Team_local_dict = Team_dict[TeamName]

    # 밑에 for문 돌리기 위한 count 설정
    count = 0

    # for문으로 url에 해당하는 웹페이지로 이동
    for link in links:
        url = f'https://gol.gg{link[2:-6]}-fullstats/'


        driver.get(url)

        time.sleep(2)

        # 버튼 클릭
        button = driver.find_element(By.ID, 'export-btn')
        
        time.sleep(1)
        
        button.click()

        time.sleep(1)

        # 클립보드 복사
        copied_df = pd.read_clipboard(sep='\t')

        # 복사한 데이터프레임 전치
        copied_df = copied_df.T

        # index를 칼럼으로 빼고
        # Player 칼럼을 index로 함.
        # CS in Enemy Jungle, Shutdown bounty collected, Shutdown bounty lost, KDA 칼럼 삭제
        # Champion, 라인 까지 칼럼에서 제외

        df = copied_df.reset_index(names='Champion')
        df = df.set_index('Player')
        df = df.drop(['CS in Enemy Jungle', 'Shutdown bounty collected', 'Shutdown bounty lost', 'KDA', 'Champion','Role'], axis=1)
        df['Solo kills'].fillna(0,inplace=True)

        

        ## % 기호가 있는 값을 가지는 칼럼에 대해 실수로 처리
        li = ['GOLD%', 'KP%', 'DMG%', 'VS%']
        for i in li:
            for j in range(len(df[i])):
                df[i][j] = float(df[i][j][:-1])
                # df.loc[j, i] = float(df.loc[j, i][:-1])

        

        # T1 각 선수 데이터 집계        
        count += 1
        for i in df.index:

            if i in Team_Player_list and i not in Team_local_dict:
                Team_local_dict[i] = pd.DataFrame(df.loc[i]).T
                Team_local_dict[i]['P value'] = result_gameclass_df.loc[count]['P value']

            elif i in Team_Player_list and i in Team_local_dict:
                Player_df = pd.DataFrame(df.loc[i]).T
                Player_df['P value'] = result_gameclass_df.loc[count]['P value']
                Team_local_dict[i] = pd.concat([Team_local_dict[i], Player_df], axis=0)
    
    # 워킹디렉토리 설정
    os.chdir('C:/Users/user/Desktop/Project2_Data/Player_data/')
    
    for playername in Team_Player_list:
        Team_local_dict[playername] = Team_local_dict[playername].astype(float)  
        # 이거 해야 데이터프레임의 숫자들이 float으로 바뀜
        
        Team_local_dict[playername].to_csv(f'{TeamName+"_"+playername}.csv') # 파일이름 ex) T1_Faker.csv
    
    
    
    
    driver.close()
    


# 경고메세지 무시
warnings.filterwarnings("ignore")

for team in Team_list:
    DataCollector(team)
    
# 경고메세지 무시 해제
warnings.filterwarnings("default")

# 학습함수

## WinLoss 함수

In [9]:
# 데이터에 win/loss를 0,1로 데이터프레임 끝에 추가하는 함수
def WinLoss(filename):
    filename['Win/Loss'] = 0
    for i in range(len(filename)):
        if filename['P value'].iloc[i]:
            filename['Win/Loss'].iloc[i] = 1
        else :
            filename['Win/Loss'].iloc[i] = 0
            
    return filename

## CVmodel 함수

In [11]:
def CVmodel(filename):
    X = filename.iloc[:,:-2]
    y = filename.iloc[:,-1]

    # 여러 파라미터들
    grid_parameters = {'max_depth' : [5,6,7,8,9,10,11],
                      'min_samples_split' : [2,3,4,5],
                      'min_samples_leaf' : [1,2,3]}   
    # 얘는 경우의 수임
    # max_depth가 1일 때 min_samples_split이                          
    # 2일때와 3일때로 작업함


    # 1. train : test = 8:2
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

    # 2. RandomForestClassifier
    randf_clf = RandomForestClassifier()


    # 2.5 GridSearchCV로 연산을 한번 함
    # cv = 3
    grid_dtree = GridSearchCV(randf_clf, param_grid = grid_parameters, cv = 5, n_jobs=-1)
    

    # 3. 적합시킨다(fitting)
    grid_dtree.fit(X_train, y_train)
    
    clf = grid_dtree.best_estimator_
    
    df = pd.DataFrame(clf.feature_importances_, index = clf.feature_names_in_).sort_values(by =0, ascending=False)[:5]
    print(df.index.values)
    
    pred = grid_dtree.best_estimator_.predict(X_test)
    
    return grid_dtree, pred

## Cleansing 함수

In [12]:
def Cleansing(filename):
    filename = WinLoss(filename)
    model, pred = CVmodel(filename)
    return model, pred

# 실행 코드

In [ ]:
# 빈 딕셔너리 만들기
Team_dict_copy = Team_dict.copy()

for team in Team_list:
    for player in Team_Player_dict[team]:
        Team_dict[team][player] = pd.read_csv(f"C:\\Users\\user\\Desktop\\Project2_Data\\Player_data\\{team}_{player}.csv", index_col=0)
        
        print(player, end = '\t')
        model, pred = Cleansing(Team_dict[team][player])
        
        Team_dict_copy[team][player] = {'model': model, 'pred' : pred}